In [1]:
%load_ext autoreload
%autoreload 2

In [55]:
from flax import nnx
import jax
import jax.random as random
import jax.numpy as jnp
import optax
from tqdm import tqdm
from utils import DataLoader

In [56]:
class first_model(nnx.Module):
    def __init__(self, n_in: int, n_mid: int, n_out: int, num_layers: int, rngs: nnx.Rngs):
        self.linear_in = nnx.Linear(n_in, n_mid, rngs = rngs)
        self.hidden_layers = ([
            nnx.Linear(n_mid, n_mid, rngs=rngs) for _ in range(num_layers)
        ])
        self.linear_out = nnx.Linear(n_mid, n_out, rngs = rngs)

    def __call__(self, x: jax.Array):
        x = self.linear_in(x)
        x = nnx.relu(x)
        for layer in self.hidden_layers:
            x = layer(x)
            x = nnx.relu(x)
        x = self.linear_out(x)
        return x

In [57]:
model = first_model(n_in = 1, n_mid = 8, n_out = 1, num_layers = 1, rngs=nnx.Rngs(0))
nnx.display(model)

In [58]:
learning_rate = 0.005
momentum = 0.9

optimizer = nnx.Optimizer(model, optax.adamw(learning_rate, momentum))
metrics_train = nnx.training.metrics.Average('loss')
metrics_test = nnx.training.metrics.Average('loss')

def loss_fn(model: first_model, x: jax.Array, y: jax.Array):
    y_pred = model(x)
    loss_per_example = optax.l2_loss(y_pred, y)
    return jnp.mean(loss_per_example)

def train_step(model: first_model, optimizer: nnx.Optimizer, metrics: nnx.Metric, x: jax.Array, y: jax.Array, loss_fn: callable):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=False)
    loss, grads = grad_fn(model, x, y)
    metrics.update(loss=loss)
    optimizer.update(grads)

train_step = nnx.jit(train_step, static_argnames=('loss_fn'))

def eval_step(model: first_model, metrics: nnx.Metric, x: jax.Array, y: jax.Array, loss_fn: callable):
  loss = loss_fn(model, x, y)
  metrics.update(loss=loss)

eval_step = nnx.jit(eval_step, static_argnames=('loss_fn'))

In [59]:
# Define dataset

train_steps = 1200
eval_every = 200
batch_size = 32
datatrain_dim = 1000
datatest_dim = 200

f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(l*x)

key = jax.random.key(0)
key, subkey = random.split(key)
toy_data = random.uniform(subkey, (datatrain_dim, 1), minval=-3, maxval=3)
toy_label = f_to_learn(0.5, 1.0, 0.1, toy_data)
train_dataloader = DataLoader(toy_data, toy_label, batch_size=batch_size, shuffle=True)

key, subkey = random.split(key)
toy_data = random.uniform(subkey, (datatest_dim, 1), minval=-3, maxval=3)
toy_label = f_to_learn(0.5, 1.0, 0.1, toy_data)
test_dataloader = DataLoader(toy_data, toy_label, batch_size=batch_size, shuffle=True)

In [60]:
epochs = 1000
pbar = tqdm(range(epochs))

for epoch in pbar:
  pbar.set_description(f"Epoch {epoch+1}")
  metrics_train.reset()
  metrics_test.reset()

  for x, y in train_dataloader:
      train_step(model=model, optimizer=optimizer, metrics=metrics_train, x=x, y=y, loss_fn=loss_fn)
  for x, y in test_dataloader:
      train_step(model=model, optimizer=optimizer, metrics=metrics_test, x=x, y=y, loss_fn=loss_fn)
    
  pbar.set_postfix({"training loss": metrics_train.compute(), "test loss": metrics_test.compute()})

Epoch 1000: 100%|██████████| 1000/1000 [03:15<00:00,  5.11it/s, training loss=0.055836976, test loss=0.06742897]
